# 📘 Lesson 6 — Gradient Boosting com XGBoost
> #### 📚 Curso Intermediate Machine Learning

- Kaggle Intermediate Machine Learning — Lesson 6
- Hybrid Learning Notebook — Study + Reference + Hands-on

<br>



> #### 🎯 Objetivos

- Nesta lesson, você aprenderá a:
    - Entender a ideia geral de **gradient boosting** como ensemble iterativo.
    - Usar `XGBRegressor` para treinar um modelo de regressão.
    - Avaliar o modelo com **MAE (Mean Absolute Error)**.
    - Ajustar parâmetros básicos:
        - `n_estimators`
        - `early_stopping_rounds`
        - `learning_rate`
        - `n_jobs`
    - Relacionar **underfitting / overfitting** com o número de árvores.
    - Manter o fluxo de trabalho consistente com as lessons anteriores.

<br>

> Este notebook segue **exatamente a lógica da lesson do Kaggle**, com explicações adicionais para estudo.

<br>

---

### 🟦 Glossário Técnico

- 
    - **Ensemble Method** — técnica que combina vários modelos para formar um modelo mais forte.
    - **Random Forest** — ensemble de árvores treinadas de forma independente, cujas previsões são combinadas por média.
    - **Gradient Boosting** — ensemble construído de forma sequencial, onde cada novo modelo corrige os erros dos anteriores.
    - **XGBoost** — implementação otimizada de gradient boosting, focada em desempenho e velocidade.
    - **XGBRegressor** — classe do XGBoost para problemas de regressão.
    - **n_estimators** — número de árvores (iterações) no ensemble.
    - **learning_rate** — fator que controla o quanto cada nova árvore contribui.
    - **early_stopping_rounds** — critério para parar o treino quando a métrica de validação não melhora.
    - **n_jobs** — número de núcleos de CPU usados em paralelo.
    - **MAE (Mean Absolute Error)** — métrica de erro usada no curso.

<br>

---


### 🟩 Mini‑Referência (API Style)

- Comandos Essenciais
    - **Importar modelo XGBoost** -> `from xgboost import XGBRegressor`
    - **Criar modelo básico** -> `my_model = XGBRegressor(random_state=0)`
    - **Definir número de árvores** -> `my_model = XGBRegressor(n_estimators=500, random_state=0)`
    - **Treinar modelo** -> `my_model.fit(X_train, y_train)`
    - **Treinar com early stopping** -> `my_model.fit(X_train, y_train, early_stopping_rounds=5, eval_set=[(X_valid, y_valid)], verbose=False)`
    - **Ajustar learning rate** -> `my_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, random_state=0)`
    - **Paralelismo** -> `my_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, n_jobs=4, random_state=0)`
    - **Avaliar com MAE**
        -  -> `from sklearn.metrics import mean_absolute_error`
        -  -> `mae = mean_absolute_error(y_valid, preds)`

<br>

---


### 🟦 Introdução — por que XGBoost aparece tanto em competições?

Ao longo do curso, usamos principalmente **Random Forest**, um método de ensemble que melhora o desempenho ao **combinar várias árvores de decisão**.
Agora entramos em outro tipo de ensemble muito popular em competições Kaggle e em projetos reais:
> - **Gradient Boosting**, na implementação **XGBoost (Extreme Gradient Boosting)**.

<br>

- Nesta lesson, o foco não é “ganhar a competição”, mas:
    - entender **o que é gradient boosting** em alto nível;
    - ver **como usar XGBRegressor** na prática;
    - aprender **quais parâmetros realmente importam** no começo;
    - conectar tudo isso com o fluxo que você já domina (Ames Housing + MAE).

<br>

---

### 🟩 Explicação conceitual — do Random Forest ao Gradient Boosting

#### 1 Random Forest (revisão rápida)

- Random Forest é um **ensemble de árvores**:
    - cada árvore é treinada de forma independente;
    - as previsões são combinadas por média;
    - reduz variância e melhora estabilidade.

- Ele já foi suficiente para:
    - comparar estratégias de **missing values**;
    - testar diferentes **encodings categóricos**;
    - integrar tudo em **pipelines** e **cross‑validation**.

---

#### 2 Gradient Boosting — ideia geral

- Gradient Boosting segue uma lógica diferente:

    1. Começa com um modelo inicial (geralmente simples).
    2. Calcula os **erros** desse modelo (via função de perda).
    3. Treina uma nova árvore focada em **corrigir esses erros**.
    4. Adiciona essa nova árvore ao ensemble.
    5. Repete o processo várias vezes.

- Cada nova árvore:
    - não tenta resolver o problema do zero;
    - foca em **ajustar o que o ensemble ainda erra**.

> O “gradient” em **gradient boosting** vem do uso de **gradiente da função de perda** para guiar como cada nova árvore deve ser ajustada.

---

#### 3 XGBoost — por que ele é tão usado?

- XGBoost é uma implementação de gradient boosting com:
    - otimizações de desempenho (CPU, memória);
    - suporte a paralelismo (`n_jobs`);
    - recursos avançados de regularização;
    - integração com scikit‑learn via `XGBRegressor`.

- Na prática:
    - é um dos modelos mais fortes para dados tabulares;
    - domina muitas competições Kaggle;
    - é relativamente simples de usar com poucos parâmetros bem escolhidos.

---

### 🟨 Implementação passo a passo (igual ao Kaggle, adaptado ao nosso ecossistema)

- Nesta seção, vamos:
    1. Carregar o dataset Ames Housing do nosso ambiente local.
    2. Preparar os dados (incluindo categóricas, como no exercise da lesson).
    3. Treinar um modelo básico com `XGBRegressor`.
    4. Avaliar com MAE.
    5. Ajustar parâmetros (`n_estimators`, `learning_rate`, `early_stopping_rounds`, `n_jobs`).

<br>

---

#### 1 Importar bibliotecas + configurar caminhos

In [ ]:
import sys
from pathlib import Path

# Caminho absoluto do notebook
notebook_path = Path().resolve()

# Sobe diretórios até encontrar config.py
for parent in notebook_path.parents:
    if (parent / "config.py").exists():
        sys.path.append(str(parent))
        break

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from config import get_data_path
DATA_PATH = get_data_path()

#### 2 Carregar dados (Ames Housing) e preparar features

Nesta lesson, o Kaggle:

- usa **tanto variáveis numéricas quanto categóricas**;
- seleciona:
    - categóricas de **baixa cardinalidade**;
    - todas as numéricas;
- aplica **one‑hot encoding** com `pd.get_dummies`.

Vamos seguir a mesma lógica.

In [3]:
# Carregar dados brutos
X_full = pd.read_csv(DATA_PATH + "train.csv", index_col="Id")
X_test_full = pd.read_csv(DATA_PATH + "test.csv", index_col="Id")

# Remover linhas sem target
X_full.dropna(axis=0, subset=["SalePrice"], inplace=True)
y = X_full.SalePrice
X_full.drop(["SalePrice"], axis=1, inplace=True)

# Separar treino/validação
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X_full, y, train_size=0.8, test_size=0.2, random_state=0
)

# Selecionar colunas categóricas de baixa cardinalidade
low_cardinality_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 10 and X_train_full[cname].dtype == "object"
]

# Selecionar colunas numéricas
numeric_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ["int64", "float64"]
]

# Manter apenas essas colunas
my_cols = low_cardinality_cols + numeric_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

# One-hot encoding com pandas (como no exercise)
X_train = pd.get_dummies(X_train)
X_valid = pd.get_dummies(X_valid)
X_test = pd.get_dummies(X_test)

# Alinhar colunas entre treino/validação/teste
X_train, X_valid = X_train.align(X_valid, join="left", axis=1)
X_train, X_test = X_train.align(X_test, join="left", axis=1)

X_train.head()

,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,GarageArea,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,MiscVal,MoSold,YrSold
Id,,,,,,,,,,,,,,,,,,,,,
619,20,90.0,11694,9,5,2007,2007,452.0,48,0,...,774,0,108,0,0,260,0,0,7,2007
871,20,60.0,6600,5,5,1962,1962,0.0,0,0,...,308,0,0,0,0,0,0,0,8,2009
93,30,80.0,13360,5,7,1921,2006,0.0,713,0,...,432,0,0,44,0,0,0,0,8,2009
818,20,NaN,13265,8,5,2002,2002,148.0,1218,0,...,857,150,59,0,0,0,0,0,7,2008
303,20,118.0,13704,7,5,2001,2002,150.0,0,0,...,843,468,81,0,0,0,0,0,1,2006


#### 3 Modelo básico com XGBRegressor (default)

- Agora vamos criar o primeiro modelo XGBoost, seguindo a lesson:

    - usar `XGBRegressor()` com parâmetros default;
    - definir `random_state=0` para reprodutibilidade;
    - treinar com `fit`;
    - avaliar com MAE.

In [9]:
from xgboost import XGBRegressor

# Modelo básico (baseline)
my_model_1 = XGBRegressor(
    random_state=0
)

my_model_1.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [10]:
# Avaliar com MAE
preds_1 = my_model_1.predict(X_valid)
mae_1 = mean_absolute_error(y_valid, preds_1)
print("MAE (modelo básico XGBoost):", mae_1)

MAE (modelo básico XGBoost): 18489.19921875


#### 4 Ajustando n_estimators — mais árvores no ensemble

O parâmetro `n_estimators` controla **quantas vezes o ciclo de boosting é repetido**, ou seja, quantas árvores entram no ensemble.

- Valores muito baixos → **underfitting** (modelo simples demais).
- Valores muito altos → risco de **overfitting** (modelo memoriza o treino).

Na lesson, valores típicos vão de 100 a 1000, dependendo de `learning_rate`.

Vamos testar um valor maior, como `n_estimators=500`.

In [12]:
my_model_2 = XGBRegressor(
    n_estimators=500,
    random_state=0
)

my_model_2.fit(X_train, y_train)

preds_2 = my_model_2.predict(X_valid)
mae_2 = mean_absolute_error(y_valid, preds_2)
print("MAE (n_estimators=500):", mae_2)

MAE (n_estimators=500): 18559.1875


#### 5 Early Stopping — deixando o modelo parar “sozinho”

- `early_stopping_rounds` permite:

    - definir um **máximo** de árvores (`n_estimators` alto);
    - deixar o modelo **parar antes** se a métrica de validação parar de melhorar.

- Na lesson, o fluxo é:

    - definir `n_estimators` relativamente alto (ex.: 500 ou 1000);
    - usar `early_stopping_rounds=5`;
    - passar `eval_set=[(X_valid, y_valid)]`;
    - definir `verbose=False` para não poluir a saída.

> - Isso ajuda a encontrar automaticamente um bom ponto de parada.

<br>

> - Neste exercicio optamos por não utilizar o `early_stopping_rounds`. <br>
O XGBRegressor da versão 3.2.0 NÃO aceita early stopping de nenhum jeito. <br> 
a unica maneira seria utilizar early stopping via API nativa. <br>
Para esta lesson não faremos early stopping de nenhum tipo, mesmo falando deles.

In [24]:
my_model_3 = XGBRegressor(
    n_estimators=500,
    random_state=0
)

my_model_3.fit(X_train, y_train)

preds_3 = my_model_3.predict(X_valid)
mae_3 = mean_absolute_error(y_valid, preds_3)
print("MAE (n_estimators=500 + early_stopping_rounds=5):", mae_3)

MAE (n_estimators=500 + early_stopping_rounds=5): 18559.1875


#### 6 Ajustando learning_rate — passos menores, mais árvores

- Em gradient boosting:

    - cada nova árvore corrige um pouco os erros anteriores;
    - `learning_rate` controla **o tamanho desse passo**.

- Se diminuímos `learning_rate`:

    - cada árvore contribui menos;
    - podemos usar **mais árvores** (`n_estimators` maior);
    - o modelo tende a ficar **mais preciso**, mas mais lento para treinar.

- Na lesson, um exemplo típico é:

    - `n_estimators=1000`
    - `learning_rate=0.05`
    - `early_stopping_rounds=5`

<br>

> - Neste exercicio optamos por não utilizar o `early_stopping_rounds`. <br>
O XGBRegressor da versão 3.2.0 NÃO aceita early stopping de nenhum jeito. <br> 
a unica maneira seria utilizar early stopping via API nativa. <br>
Para esta lesson não faremos early stopping de nenhum tipo, mesmo falando deles

In [20]:
my_model_4 = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=0
)

my_model_4.fit(X_train, y_train)

preds_4 = my_model_4.predict(X_valid)
mae_4 = mean_absolute_error(y_valid, preds_4)
print("MAE:", mae_4)

MAE: 17978.990234375


#### 7 n_jobs — usando múltiplos núcleos de CPU

Para datasets maiores, o treino pode demorar.  
O parâmetro `n_jobs` permite usar vários núcleos de CPU em paralelo.

- Em máquinas locais, é comum usar `n_jobs` igual ao número de cores.
- Em datasets pequenos, o ganho é pequeno, mas o código fica pronto para escalar.

> Importante: isso **não muda a qualidade do modelo**, apenas o tempo de treino.

<br>

> - Neste exercicio optamos por não utilizar o `early_stopping_rounds`. <br>
O XGBRegressor da versão 3.2.0 NÃO aceita early stopping de nenhum jeito. <br> 
a unica maneira seria utilizar early stopping via API nativa. <br>
Para esta lesson não faremos early stopping de nenhum tipo, mesmo falando deles

In [22]:
my_model_5 = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    n_jobs=4,
    random_state=0
)

my_model_5.fit(X_train, y_train)

preds_5 = my_model_5.predict(X_valid)
mae_5 = mean_absolute_error(y_valid, preds_5)
print("MAE (n_estimators=1000, lr=0.05, n_jobs=4, early_stopping=5):", mae_5)

MAE (n_estimators=1000, lr=0.05, n_jobs=4, early_stopping=5): 17978.990234375


#### 8 Comparação rápida dos modelos testados

In [26]:
results = {
    "Modelo 1 — default": mae_1,
    "Modelo 2 — 500 árvores": mae_2,
    "Modelo 3 — 500": mae_3,
    "Modelo 4 — 1000 + lr=0.05": mae_4,
    "Modelo 5 — 1000 + lr=0.05 + n_jobs=4": mae_5,
}

for name, mae in results.items():
    print(f"{name}: {mae}")

Modelo 1 — default: 18489.19921875
Modelo 2 — 500 árvores: 18559.1875
Modelo 3 — 500: 18559.1875
Modelo 4 — 1000 + lr=0.05: 17978.990234375
Modelo 5 — 1000 + lr=0.05 + n_jobs=4: 17978.990234375


### 🟪 Observações pedagógicas do Copilot (o que o Kaggle assume)

1. **XGBoost não substitui o pré‑processamento**  
   - Aqui usamos:
     - seleção de colunas (numéricas + categóricas de baixa cardinalidade);
     - one‑hot encoding;
     - alinhamento de colunas entre treino/validação/teste.
   - Tudo isso vem das lessons anteriores (Missing Values, Categorical Variables).

2. **Parâmetros que realmente importam no começo**
   - `n_estimators`: controla o tamanho do ensemble.
   - `learning_rate`: controla o tamanho do passo de correção.
   - `early_stopping_rounds`: evita overfitting e economiza tempo.
   - `n_jobs`: ajuda em datasets maiores.

3. **Como pensar em underfitting / overfitting aqui**
   - `n_estimators` muito baixo → modelo simples demais.
   - `n_estimators` muito alto sem early stopping → risco de overfitting.
   - `learning_rate` muito alto → passos grandes demais, modelo instável.
   - `learning_rate` menor + mais árvores → geralmente melhor, mas mais lento.

4. **XGBoost não é mágica**
   - Ele é poderoso, mas:
     - ainda depende de bons dados;
     - ainda precisa de validação correta (Cross‑Validation, quando fizer sentido);
     - ainda pode sofrer com leakage, se o fluxo de dados estiver errado.

5. **Conexão com Cross‑Validation (Lesson 05)**
   - Tudo o que você aprendeu sobre:
     - dividir dados corretamente;
     - usar métricas como MAE;
     - comparar alternativas com base em validação;
   - continua valendo aqui — XGBoost é “só” mais um modelo dentro do mesmo fluxo.

<br>

---

### 🟧 Conclusão — o que levar desta lesson

- Você viu **o que é gradient boosting** em alto nível:
  - ensemble sequencial;
  - cada árvore corrige os erros das anteriores.

- Conheceu o **XGBoost (XGBRegressor)**:
  - implementação otimizada de gradient boosting;
  - integração simples com scikit‑learn.

- Aprendeu a usar e interpretar os principais parâmetros:
  - `n_estimators` — número de árvores;
  - `learning_rate` — tamanho do passo;
  - `early_stopping_rounds` — parada automática;
  - `n_jobs` — paralelismo.

- Manteve o fluxo consistente com as lessons anteriores:
  - Ames Housing;
  - seleção de colunas;
  - one‑hot encoding;
  - avaliação com MAE.

> Próximo passo (exercise do Kaggle):  
> - treinar seus próprios modelos XGBoost;  
> - experimentar combinações de parâmetros;  
> - observar na prática como eles afetam o MAE.